In [1]:
!pip install ultralytics opencv-python


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\igorc\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np

model = YOLO("yolo11s-pose.pt")

cap = cv2.VideoCapture(0)

exercicios = [
    {
        "nome": "Elevacao Lateral",
        "meta": 5
    },
    {
        "nome": "Ponte",
        "meta": 5
    },
    {
        "nome": "Agachamento",
        "meta": 5
    }
]

exercicio_atual = 0
repeticoes = 0
estado = "baixo"

def calcular_distancia(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model.track(
        frame,
        persist=True,
        conf=0.5
    )

    annotated_frame = results[0].plot(
        boxes=False,
        labels=False
    )

    nome_exercicio = exercicios[exercicio_atual]["nome"]
    meta = exercicios[exercicio_atual]["meta"]

    if results[0].keypoints is not None:

        pontos = results[0].keypoints.xy.cpu().numpy()

        if len(pontos) > 0:

            pessoa = pontos[0]

            ombro = pessoa[5]
            quadril = pessoa[11]
            joelho = pessoa[13]
            punho = pessoa[9]

            # ==========================
            # EXERCÍCIO 1 — PONTE
            # ==========================

            if nome_exercicio == "Ponte":

                altura_quadril = quadril[1]

                if altura_quadril < 300 and estado == "baixo":
                    estado = "alto"

                if altura_quadril > 350 and estado == "alto":
                    repeticoes += 1
                    estado = "baixo"

            # ==========================
            # EXERCÍCIO 2 — AGACHAMENTO
            # ==========================

            elif nome_exercicio == "Agachamento":

                altura_quadril = quadril[1]

                if altura_quadril > 350 and estado == "alto":
                    estado = "baixo"

                if altura_quadril < 300 and estado == "baixo":
                    repeticoes += 1
                    estado = "alto"

            # ==========================
            # EXERCÍCIO 3 — ELEVAÇÃO
            # ==========================

            elif nome_exercicio == "Elevacao Lateral":

                altura_punho = punho[1]
                altura_ombro = ombro[1]

                if altura_punho < altura_ombro and estado == "baixo":
                    estado = "alto"

                if altura_punho > altura_ombro and estado == "alto":
                    repeticoes += 1
                    estado = "baixo"

    # ==========================
    # TROCA DE EXERCÍCIO
    # ==========================

    if repeticoes >= meta:

        exercicio_atual += 1
        repeticoes = 0
        estado = "baixo"

        if exercicio_atual >= len(exercicios):
            exercicio_atual = len(exercicios) - 1

    # ==========================
    # INTERFACE
    # ==========================

    cv2.rectangle(
        annotated_frame,
        (10, 10),
        (500, 120),
        (0, 0, 0),
        -1
    )

    overlay = annotated_frame.copy()

    cv2.rectangle(
        overlay,
        (10, 10),
        (500, 120),
        (0, 0, 0),
        -1
    )

    alpha = 0.5

    cv2.addWeighted(
        overlay,
        alpha,
        annotated_frame,
        1 - alpha,
        0,
        annotated_frame
    )

    cv2.putText(
        annotated_frame,
        f"Exercicio: {exercicios[exercicio_atual]['nome']}",
        (20, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2
    )

    cv2.putText(
        annotated_frame,
        f"Repeticoes: {repeticoes}/{meta}",
        (20, 100),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 0),
        2
    )

    cv2.imshow("Fisioterapia IA", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


0: 480x640 (no detections), 610.6ms
Speed: 12.0ms preprocess, 610.6ms inference, 12.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 695.9ms
Speed: 4.7ms preprocess, 695.9ms inference, 3.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 602.9ms
Speed: 79.3ms preprocess, 602.9ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 434.4ms
Speed: 8.1ms preprocess, 434.4ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 383.9ms
Speed: 6.7ms preprocess, 383.9ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 400.7ms
Speed: 9.7ms preprocess, 400.7ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 366.7ms
Speed: 3.6ms preprocess, 366.7ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 451.3ms
Speed: 2.3ms pr